In [12]:
# ==============================
# MASTER SETUP CELL (RUN FIRST)
# ==============================

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from datetime import datetime, timedelta

# -------- BROKER CONFIG --------
API_KEY = "YOUR_API_KEY"
ACCESS_TOKEN = "YOUR_ACCESS_TOKEN"
BASE_URL = "https://api.yourbroker.com"   # <-- THIS WAS MISSING

# -------- STRATEGY CONFIG ------
SYMBOL = "NIFTY"
INTERVAL = "15minute"

ORB_START = "09:15"
ORB_END = "09:30"
EXIT_TIME = "15:15"

LOT_SIZE = 50
TARGET_POINTS = 30
STOPLOSS_POINTS = 15

TRADE_TAKEN = False

print("✅ Environment initialized")


In [13]:
def get_candles(symbol, interval, start, end):
    """
    Fetch OHLC candle data from broker API
    """
    url = f"{BASE_URL}/market/candles"
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
    params = {
        "symbol": symbol,
        "interval": interval,
        "from": start,
        "to": end
    }
    
    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    df = pd.DataFrame(
        data["candles"],
        columns=["time", "open", "high", "low", "close", "volume"]
    )
    return df


def get_ltp(symbol):
    url = f"{BASE_URL}/market/ltp"
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
    response = requests.get(url, headers=headers, params={"symbol": symbol})
    return response.json()["ltp"]


def place_order(symbol, side, quantity):
    """
    MARKET order
    """
    url = f"{BASE_URL}/orders/place"
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

    payload = {
        "symbol": symbol,
        "side": side,
        "quantity": quantity,
        "order_type": "MARKET",
        "product": "MIS"
    }

    response = requests.post(url, headers=headers, json=payload)
    return response.json()

In [14]:
def calculate_orb():
    today = datetime.now().strftime("%Y-%m-%d")

    start_time = f"{today} {ORB_START}"
    end_time = f"{today} {ORB_END}"

    df = get_candles(SYMBOL, INTERVAL, start_time, end_time)

    orb_high = df["high"].max()
    orb_low = df["low"].min()

    print(f"ORB High: {orb_high}")
    print(f"ORB Low : {orb_low}")

    return orb_high, orb_low

In [15]:
def get_atm_option(direction):
    """
    Simple ATM option selection logic
    """
    spot = get_ltp(SYMBOL)
    strike = round(spot / 50) * 50

    expiry = "24JAN"  # CHANGE TO CURRENT WEEKLY EXPIRY

    if direction == "CALL":
        return f"NIFTY{expiry}{strike}CE"
    else:
        return f"NIFTY{expiry}{strike}PE"


In [16]:
def manage_trade(option_symbol, entry_price):
    target = entry_price + TARGET_POINTS
    stoploss = entry_price - STOPLOSS_POINTS

    print(f"Target   : {target}")
    print(f"Stoploss : {stoploss}")

    while True:
        ltp = get_ltp(option_symbol)

        if ltp >= target:
            place_order(option_symbol, "SELL", LOT_SIZE)
            print("🎯 Target Hit")
            break

        elif ltp <= stoploss:
            place_order(option_symbol, "SELL", LOT_SIZE)
            print("🛑 Stoploss Hit")
            break

        time.sleep(1)


In [17]:
def run_orb_strategy():
    global TRADE_TAKEN

    orb_high, orb_low = calculate_orb()

    print("Waiting for breakout...")

    while datetime.now().time() < datetime.strptime(EXIT_TIME, "%H:%M").time():

        if TRADE_TAKEN:
            time.sleep(5)
            continue

        spot_ltp = get_ltp(SYMBOL)

        # CALL BUY
        if spot_ltp > orb_high:
            option = get_atm_option("CALL")
            entry_price = get_ltp(option)

            place_order(option, "BUY", LOT_SIZE)
            TRADE_TAKEN = True

            print(f"📈 CALL Bought: {option} @ {entry_price}")
            manage_trade(option, entry_price)

        # PUT BUY
        elif spot_ltp < orb_low:
            option = get_atm_option("PUT")
            entry_price = get_ltp(option)

            place_order(option, "BUY", LOT_SIZE)
            TRADE_TAKEN = True

            print(f"📉 PUT Bought: {option} @ {entry_price}")
            manage_trade(option, entry_price)

        time.sleep(2)


In [18]:
print("🚀 Starting 15-Min ORB Strategy")
run_orb_strategy()


🚀 Starting 15-Min ORB Strategy


NameError: name 'BASE_URL' is not defined